# Making the JEV router cheaper

The JEV router in `adaptive.ipynb` (Qwen3.5-4B, bf16) routed all 30 test queries correctly but costs ~175 ms,
~198 prompt tokens and ~8 GB of GPU memory per query, against 14 ms / 12 tokens / 1.3 GB for the fine-tuned BERT router.
This notebook tries four ways to close that gap, on the same 30 held-out queries and with the same timing method
(one query at a time, 3 warm-up calls, CUDA synchronised before and after each call):

1. **Prefix caching** – the instruction and route descriptions are identical for every query, so their KV cache is
   computed once and reused; only the query and the answer options are processed per call.
2. **Smaller models** – Qwen3.5-0.8B instead of 4B (same family, same prompt), plus
   Qwen2.5-1.5B-Instruct (Qwen3.5 has no 1.5B size; this one is from the older Qwen2.5 family).
3. **Batching** – all 30 queries in one `decide_many` call, so fixed per-call costs are shared.
4. **4-bit quantization** – the 4B and 1.5B models in NF4 via bitsandbytes.

Every model is run in all three call modes below, so the 1.5B model is tested with each technique and with 4-bit
combined with prefix caching and with batching.

The prompt is exactly the one used in `adaptive.ipynb`, so accuracy differences come from the model, not the wording.

In [1]:

import copy
import gc
import json
import statistics
import time
from pathlib import Path

import pandas as pd
import torch
from so1 import Choice, Decider
from transformers import BitsAndBytesConfig

ROUTES = ["direct_answer", "web_search", "rag_retrieval"]
ROUTE_DESCRIPTIONS = """You are routing a user query to one of three routes:
- direct_answer: simple queries the LLM can answer from general knowledge (greetings, common facts, general definitions, simple math).
- web_search: needs up-to-date information from the internet (news, prices, recent events, current versions).
- rag_retrieval: asks about RAG and retrieval research covered by the internal paper collection (e.g. DPR, RAG, HyDE, FLARE, Self-RAG, CRAG, RAPTOR, Adaptive-RAG), or needs deep, detailed analysis of those documents, even if the question is short."""
CHOICE = Choice("Which route should handle this query?", ROUTES)
THRESHOLD = 0.7  # same fallback as the pipeline: below this confidence the query goes to rag_retrieval

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
test = json.loads(Path("eval/router_dataset.json").read_text(encoding="utf-8"))["test"]
labels = [row["label"] for row in test]
queries = [row["query"] for row in test]


def state_for(query):
    return f"{ROUTE_DESCRIPTIONS}\n\nUser query: {query}"


def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def per_query_latency(route_fn, warmup=3):
    """Route each query alone; return (choices, confidences, latencies in ms)."""
    for q in queries[:warmup]:
        route_fn(q)
    choices, confidences, ms = [], [], []
    for q in queries:
        sync()
        start = time.perf_counter()
        choice, confidence = route_fn(q)
        sync()
        ms.append((time.perf_counter() - start) * 1000)
        choices.append(choice)
        confidences.append(confidence)
    return choices, confidences, ms


def summarize(model_name, mode, choices, confidences, ms, prompt_tokens, computed_tokens, extra):
    ms_sorted = sorted(ms)
    return {
        "model": model_name,
        "mode": mode,
        "accuracy": sum(c == l for c, l in zip(choices, labels)) / len(labels),
        "accuracy_after_fallback": sum((c if f >= THRESHOLD else "rag_retrieval") == l
                                       for c, f, l in zip(choices, confidences, labels)) / len(labels),
        "fallback_count": sum(f < THRESHOLD for f in confidences),
        "misrouted": [q for q, c, l in zip(queries, choices, labels) if c != l],
        "mean_confidence": statistics.mean(confidences),
        "min_confidence": min(confidences),
        "mean_ms": statistics.mean(ms),
        "median_ms": statistics.median(ms),
        "p95_ms": ms_sorted[round(0.95 * (len(ms) - 1))],
        "total_s": sum(ms) / 1000,
        "prompt_tokens_per_query": prompt_tokens,
        "computed_tokens_per_query": computed_tokens,
        **extra,
        "rows": [{"query": q, "label": l, "predicted": c, "confidence": f, "latency_ms": m}
                 for q, l, c, f, m in zip(queries, labels, choices, confidences, ms)],
    }


## The three ways of calling the model

- **full** – `decider.decide(...)`, exactly what `route_query` in `adaptive.ipynb` does.
- **prefix-cached** – the shared prompt prefix (chat header, instruction, route descriptions, `User query: `) is run
  through the model once; each call deep-copies that KV cache and feeds only the remaining tokens. The prefix is found
  by tokenizing two different queries and taking their common leading tokens, so the token sequence the model sees is
  identical to the full prompt.
- **batched** – all 30 queries in one `decide_many` call (batch size 32). Reported per query = total time / 30.

In [2]:

class PrefixCachedRouter:
    """JEV routing with the shared prompt prefix computed once and reused for every query."""

    def __init__(self, decider):
        self.decider = decider
        self.model = decider.backend.model
        self.device = decider.backend.device
        self.label_ids = list(decider.prompts.label_ids(len(ROUTES)))
        a, b = self.prompt_ids("alpha"), self.prompt_ids("zebra quantum")
        self.prefix_len = next(i for i, (x, y) in enumerate(zip(a, b)) if x != y)
        self.prefix = a[: self.prefix_len]
        with torch.inference_mode():
            ids = torch.tensor([self.prefix], device=self.device)
            self.cache = self.model(input_ids=ids, use_cache=True).past_key_values

    def prompt_ids(self, query):
        return self.decider.prompts.separate(state_for(query), [CHOICE])[0].ids

    @torch.inference_mode()
    def __call__(self, query):
        ids = self.prompt_ids(query)
        assert ids[: self.prefix_len] == self.prefix, "query changed the shared prefix"
        cache = copy.deepcopy(self.cache)  # the forward pass extends the cache in place
        rest = torch.tensor([ids[self.prefix_len:]], device=self.device)
        logits = self.model(input_ids=rest, past_key_values=cache, use_cache=True).logits
        probs = logits[0, -1, self.label_ids].float().softmax(-1)
        i = int(probs.argmax())
        return ROUTES[i], float(probs[i])

    def computed_tokens(self, query):
        return len(self.prompt_ids(query)) - self.prefix_len


def benchmark_model(model_name, label, **load_kwargs):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    decider = Decider.from_pretrained(model_name, backend="hf", **load_kwargs)
    load_s = time.perf_counter() - start
    weights_mb = torch.cuda.memory_allocated() / 2**20
    model = decider.backend.model
    extra = {
        "load_s": load_s,
        "weights_gpu_mb": weights_mb,
        "parameters_m": sum(p.numel() for p in model.parameters()) / 1e6,
        "devices": sorted({str(d) for d in getattr(model, "hf_device_map", {}).values()}) or [str(decider.backend.device)],
    }
    prompt_tokens = statistics.mean(len(decider.prompts.separate(state_for(q), [CHOICE])[0].ids) for q in queries)
    rows = []

    def full(q):
        d = decider.decide(state_for(q), [CHOICE], mode="separate")[0]
        return d.choice, d.confidence

    torch.cuda.reset_peak_memory_stats()
    c, conf, ms = per_query_latency(full)
    rows.append(summarize(label, "full", c, conf, ms, prompt_tokens, prompt_tokens,
                          {**extra, "peak_gpu_mb": torch.cuda.max_memory_allocated() / 2**20}))

    cached = PrefixCachedRouter(decider)
    torch.cuda.reset_peak_memory_stats()
    c, conf, ms = per_query_latency(cached)
    computed = statistics.mean(cached.computed_tokens(q) for q in queries)
    rows.append(summarize(label, "prefix-cached", c, conf, ms, prompt_tokens, computed,
                          {**extra, "peak_gpu_mb": torch.cuda.max_memory_allocated() / 2**20, "prefix_tokens": cached.prefix_len}))

    decider.backend.batch_size = 32
    items = [(state_for(q), [CHOICE]) for q in queries]
    decider.decide_many(items[:4], mode="separate")  # warm-up
    torch.cuda.reset_peak_memory_stats()
    sync()
    start = time.perf_counter()
    decisions = decider.decide_many(items, mode="separate")
    sync()
    total_ms = (time.perf_counter() - start) * 1000
    c = [d[0].choice for d in decisions]
    conf = [d[0].confidence for d in decisions]
    per = [total_ms / len(queries)] * len(queries)
    rows.append(summarize(label, "batched (30 in one call)", c, conf, per, prompt_tokens, prompt_tokens,
                          {**extra, "peak_gpu_mb": torch.cuda.max_memory_allocated() / 2**20, "batch_total_ms": total_ms}))

    del cached, decider, model
    gc.collect()
    torch.cuda.empty_cache()
    return rows


## Run every model

In [3]:

NF4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
VARIANTS = [
    ("Qwen/Qwen3.5-4B", "Qwen3.5-4B bf16", {"dtype": torch.bfloat16}),
    ("Qwen/Qwen3.5-4B", "Qwen3.5-4B 4-bit NF4", {"dtype": torch.bfloat16, "quantization_config": NF4}),
    # ("Qwen/Qwen3.5-2B", "Qwen3.5-2B bf16", {"dtype": torch.bfloat16}),  # not run: 4.5 GB download stalled
    ("Qwen/Qwen2.5-1.5B-Instruct", "Qwen2.5-1.5B bf16 (older family)", {"dtype": torch.bfloat16}),
    ("Qwen/Qwen2.5-1.5B-Instruct", "Qwen2.5-1.5B 4-bit NF4", {"dtype": torch.bfloat16, "quantization_config": NF4}),
    ("Qwen/Qwen3.5-0.8B", "Qwen3.5-0.8B bf16", {"dtype": torch.bfloat16}),
]

all_rows = []
for model_name, label, kwargs in VARIANTS:
    try:
        rows = benchmark_model(model_name, label, **kwargs)
    except Exception as e:  # record the failure instead of losing the other variants
        print(f"{label}: FAILED - {type(e).__name__}: {e}")
        all_rows.append({"model": label, "mode": "failed", "error": f"{type(e).__name__}: {e}"})
        gc.collect()
        torch.cuda.empty_cache()
        continue
    all_rows += rows
    for r in rows:
        print(f'{label:<34} {r["mode"]:<26} acc {r["accuracy"]:.3f} (after fallback {r["accuracy_after_fallback"]:.3f})  {r["mean_ms"]:7.1f} ms/query  '
              f'{r["computed_tokens_per_query"]:6.1f} tok computed  peak {r["peak_gpu_mb"]:7.0f} MB')

(RESULTS / "router_jev_variants.json").write_text(json.dumps(all_rows, indent=2), encoding="utf-8")


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Qwen3.5-4B bf16                    full                       acc 1.000 (after fallback 1.000)    156.3 ms/query   198.1 tok computed  peak    8118 MB
Qwen3.5-4B bf16                    prefix-cached              acc 1.000 (after fallback 1.000)    147.7 ms/query    46.1 tok computed  peak    8198 MB
Qwen3.5-4B bf16                    batched (30 in one call)   acc 1.000 (after fallback 1.000)     64.4 ms/query   198.1 tok computed  peak   10046 MB


W1021 02:23:40.531000 9244 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2.5-1.5B 4-bit NF4             full                       acc 0.700 (after fallback 0.633)     77.8 ms/query   208.1 tok computed  peak    1223 MB
Qwen2.5-1.5B 4-bit NF4             prefix-cached              acc 0.700 (after fallback 0.667)     92.4 ms/query    40.1 tok computed  peak    1228 MB
Qwen2.5-1.5B 4-bit NF4             batched (30 in one call)   acc 0.700 (after fallback 0.633)     15.1 ms/query   208.1 tok computed  peak    1608 MB


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Qwen3.5-0.8B bf16                  full                       acc 0.933 (after fallback 0.400)    112.5 ms/query   198.1 tok computed  peak    1498 MB
Qwen3.5-0.8B bf16                  prefix-cached              acc 0.967 (after fallback 0.367)    108.6 ms/query    46.1 tok computed  peak    1535 MB
Qwen3.5-0.8B bf16                  batched (30 in one call)   acc 0.967 (after fallback 0.400)     16.1 ms/query   198.1 tok computed  peak    2419 MB


119766

## Comparison with the original JEV and the fine-tuned BERT router

In [4]:

table = pd.DataFrame([r for r in all_rows if r["mode"] != "failed"])
bert = json.loads((RESULTS / "router_bert.json").read_text(encoding="utf-8"))["summary"]
bert_row = {
    "model": "BERT (fine-tuned)", "mode": "full", "accuracy": bert["accuracy"],
    "accuracy_after_fallback": bert["accuracy_after_fallback"], "fallback_count": bert["fallback_count"],
    "mean_ms": bert["latency"]["mean_ms"], "median_ms": bert["latency"]["median_ms"], "p95_ms": bert["latency"]["p95_ms"],
    "prompt_tokens_per_query": bert["tokens"]["mean_input_per_query"],
    "computed_tokens_per_query": bert["tokens"]["mean_input_per_query"],
    "weights_gpu_mb": None, "peak_gpu_mb": bert["peak_gpu_memory_mb"], "parameters_m": bert["parameters"] / 1e6,
}
table = pd.concat([table, pd.DataFrame([bert_row])], ignore_index=True)
cols = ["model", "mode", "accuracy", "accuracy_after_fallback", "fallback_count", "mean_ms", "median_ms", "p95_ms", "prompt_tokens_per_query",
        "computed_tokens_per_query", "parameters_m", "weights_gpu_mb", "peak_gpu_mb"]
baseline = table[(table.model == "Qwen3.5-4B bf16") & (table["mode"] == "full")]["mean_ms"].iloc[0]
table["speedup_vs_4b_full"] = baseline / table["mean_ms"]
display(table[cols + ["speedup_vs_4b_full"]].round(2))
for r in all_rows:
    if r.get("misrouted"):
        print(f'{r["model"]} / {r["mode"]} misrouted: {r["misrouted"]}')


,model,mode,accuracy,accuracy_after_fallback,fallback_count,mean_ms,median_ms,p95_ms,prompt_tokens_per_query,computed_tokens_per_query,parameters_m,weights_gpu_mb,peak_gpu_mb,speedup_vs_4b_full
0,Qwen3.5-4B bf16,full,1.00,1.00,0,156.33,155.11,175.25,198.10,198.10,4205.75,8021.861328,8118.43,1.00
1,Qwen3.5-4B bf16,prefix-cached,1.00,1.00,0,147.74,145.81,163.33,198.10,46.10,4205.75,8021.861328,8197.90,1.06
2,Qwen3.5-4B bf16,batched (30 in one call),1.00,1.00,0,64.36,64.36,64.36,198.10,198.10,4205.75,8021.861328,10046.04,2.43
3,Qwen3.5-4B 4-bit NF4,full,1.00,1.00,0,194.81,192.71,212.59,198.10,198.10,2421.21,3203.029297,3267.69,0.80
4,Qwen3.5-4B 4-bit NF4,prefix-cached,1.00,1.00,0,173.85,172.99,192.74,198.10,46.10,2421.21,3203.029297,3351.57,0.90
5,Qwen3.5-4B 4-bit NF4,batched (30 in one call),1.00,1.00,0,64.57,64.57,64.57,198.10,198.10,2421.21,3203.029297,5201.83,2.42
6,Qwen2.5-1.5B bf16 (older family),full,0.77,0.73,2,55.93,54.54,71.24,208.13,208.13,1543.71,2976.402344,2990.18,2.79
7,Qwen2.5-1.5B bf16 (older family),prefix-cached,0.77,0.70,3,62.59,62.39,71.73,208.13,40.13,1543.71,2976.402344,3001.27,2.50
8,Qwen2.5-1.5B bf16 (older family),batched (30 in one call),0.73,0.70,3,15.00,15.00,15.00,208.13,208.13,1543.71,2976.402344,3395.77,10.42
9,Qwen2.5-1.5B 4-bit NF4,full,0.70,0.63,4,77.83,78.20,87.89,208.13,208.13,888.62,1189.841797,1223.09,2.01


Qwen2.5-1.5B bf16 (older family) / full misrouted: ['Who wrote the novel 1984?', 'What is a database index?', 'What is the difference between weather and climate?', 'Convert 2 hours into seconds.', 'What is supervised learning?', 'What are the latest features in the newest Python release?', 'Which AI models were released this week?']
Qwen2.5-1.5B bf16 (older family) / prefix-cached misrouted: ['Who wrote the novel 1984?', 'What is a database index?', 'What is the difference between weather and climate?', 'Convert 2 hours into seconds.', 'What is supervised learning?', 'What are the latest features in the newest Python release?', 'Which AI models were released this week?']
Qwen2.5-1.5B bf16 (older family) / batched (30 in one call) misrouted: ['Who wrote the novel 1984?', 'What is a database index?', 'What does GPU stand for?', 'What is the difference between weather and climate?', 'Convert 2 hours into seconds.', 'What is supervised learning?', 'What are the latest features in the newe